#Imports

In [1]:
import re
import pandas as pd

#Jan 2017

In [23]:
import re
import pandas as pd

x  = "/content/phivolcs_2017_January.txt"
y = "phivolcs_earthquakes_2017_01_January.csv"

#patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

records = []
state = 0#for header
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        if state == 0:
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue
        if state == 1:# latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue
        if state == 2:# longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0#reset here after format break
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:#magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:#location
            cur["Location"] = line
            records.append(cur)
            state = 0
            cur = {}


df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])


for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
#--separation
df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]
#---

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())

#----save---#
df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {OUTPUT_FILE}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         0
dtype: int64
Data Saved: 341 rows → phivolcs_earthquakes(2017_January).csv


In [18]:
df.head(10)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,31 Jan 2017 - 11:55 PM,5.13,126.10,14,3.5,073 km S 67° E,Sarangani (Davao Occidental)
1,31 Jan 2017 - 11:42 PM,9.82,125.50,37,2.5,003 km N 16° E,Surigao City (Surigao Del Norte)
2,31 Jan 2017 - 05:17 PM,9.23,122.57,19,3.7,022 km S 21° W,Basay (Negros Oriental)
3,31 Jan 2017 - 05:26 AM,8.78,123.27,14,2.0,022 km N 20° W,Dipolog City (Zamboanga del Norte)
4,31 Jan 2017 - 05:19 AM,13.72,120.56,117,2.3,015 km S 33° W,Calatagan (Batangas)
5,31 Jan 2017 - 04:32 AM,5.20,126.10,55,3.2,074 km S 69° E,Sarangani (Davao Occidental)
6,31 Jan 2017 - 02:34 AM,9.98,122.24,19,2.7,042 km N 89° W,Cauayan (Negros Occidental)
7,30 Jan 2017 - 11:19 PM,5.91,125.29,18,1.8,007 km S 05° E,Malapatan (Sarangani)
8,30 Jan 2017 - 11:16 PM,9.87,124.12,34,2.0,006 km S 28° E,Sagbayan (Bohol)
9,30 Jan 2017 - 10:33 PM,13.52,124.58,20,2.5,028 km S 56° E,Baras (Catanduanes)


#Feb 2017

In [24]:
import re
import pandas as pd

x  = "/content/2017feb.txt"
y = "phivolcs_earthquakes_2017_02_Febuary.csv"

#patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

records = []
state = 0#for header
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        if state == 0:
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue
        if state == 1:# latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue
        if state == 2:# longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0#reset here after format break
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:#magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:#location
            cur["Location"] = line
            records.append(cur)
            state = 0
            cur = {}


df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])


for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
#--separation
df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]
#---

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())

#----save---#
df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {OUTPUT_FILE}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         0
dtype: int64
Data Saved: 383 rows → phivolcs_earthquakes(2017_January).csv


In [25]:
df.head(10)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,28 Feb 2017 - 11:26 PM,14.05,120.42,116,2.4,022 km S 81° W,Nasugbu (Batangas)
1,28 Feb 2017 - 11:15 PM,6.22,125.24,106,2.4,005 km S 15° W,Malungon (Sarangani)
2,28 Feb 2017 - 11:00 PM,17.90,120.78,14,3.5,039 km S 31° E,Laoag City
3,28 Feb 2017 - 09:36 PM,13.52,124.39,15,2.2,019 km S 67° E,Virac (Catanduanes)
4,28 Feb 2017 - 07:56 PM,4.51,125.57,14,2.5,103 km S 06° E,Sarangani (Davao Occidental)
5,28 Feb 2017 - 01:02 PM,4.53,127.21,84,3.5,217 km S 62° E,Sarangani (Davao Occidental)
6,28 Feb 2017 - 11:26 AM,12.16,125.87,17,3.3,039 km S 86° E,San Policarpo (Eastern Samar)
7,28 Feb 2017 - 05:43 AM,18.62,121.20,14,2.8,013 km N 86° E,Claveria (Cagayan)
8,28 Feb 2017 - 05:18 AM,16.87,120.19,16,2.3,023 km N 46° W,Bacnotan (La Union)
9,28 Feb 2017 - 03:44 AM,13.94,120.43,88,2.1,025 km N 62° W,Calatagan (Batangas)


#Mar 2017

In [26]:
import re
import pandas as pd

x  = "/content/2017mar.txt"
y = "phivolcs_earthquakes_2017_03_March.csv"

#patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

records = []
state = 0#for header
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        if state == 0:
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue
        if state == 1:# latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue
        if state == 2:# longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0#reset here after format break
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:#magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:#location
            cur["Location"] = line
            records.append(cur)
            state = 0
            cur = {}


df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])


for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
#--separation
df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]
#---

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())

#----save---#
df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {OUTPUT_FILE}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         0
dtype: int64
Data Saved: 366 rows → phivolcs_earthquakes(2017_January).csv


In [27]:
df.head(10)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,31 Mar 2017 - 08:53 PM,9.80,122.51,30,2.5,009 km S 81° W,Candoni (Negros Occidental)
1,31 Mar 2017 - 08:25 PM,9.97,126.23,11,3.1,016 km N 78° E,San Isidro (Surigao del Norte)
2,31 Mar 2017 - 07:00 PM,13.60,120.24,5,2.7,014 km S 06° W,Looc (Occidental Mindoro)
3,31 Mar 2017 - 06:52 PM,19.78,120.25,43,4.4,139 km N 67° W,Calayan (Cagayan)
4,31 Mar 2017 - 06:29 PM,12.95,124.67,24,2.1,051 km N 34° W,Pambujan (Northern Samar)
5,31 Mar 2017 - 06:08 PM,10.21,125.49,31,2.1,020 km S 30° W,Loreto (Dinagat)
6,31 Mar 2017 - 05:15 PM,8.66,122.75,13,2.4,041 km N 68° W,Manukan (Zamboanga del Norte)
7,31 Mar 2017 - 04:32 PM,16.61,119.95,22,2.0,025 km N 15° E,Bolinao (Pangasinan)
8,31 Mar 2017 - 04:29 PM,13.04,124.67,34,2.0,044 km S 88° E,Prieto Diaz (Sorsogon)
9,31 Mar 2017 - 01:44 PM,13.73,120.57,127,2.8,013 km S 30° W,Calatagan (Batangas)


#April 2017

In [61]:
import re
import pandas as pd

x  = "/content/2017apr.txt"
y = "phivolcs_earthquakes_2017_04_April.csv"

#patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

records = []
state = 0#for header
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        if state == 0:
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue
        if state == 1:# latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue
        if state == 2:# longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0#reset here after format break
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:#magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            if line:  # only save non-empty
              cur["Location"] = line
              records.append(cur)
              state = 0
              cur = {}


df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])


for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
#--separation
df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]
#---

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())

#----save---#
df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {OUTPUT_FILE}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         7
dtype: int64
Data Saved: 939 rows → phivolcs_earthquakes(2017_January).csv


In [62]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
133,25 Apr 2017 - 04:56 AM,11.90,124.05,1,3.0,014 km S 25° E Cataingan (Masbate),None
134,25 Apr 2017 - 02:48 AM,9.44,125.63,19,2.6,007 km S 68° E Kitcharao (Agusan Del Norte),None
135,25 Apr 2017 - 02:35 AM,13.90,120.86,28,2.4,008 km N 52° W San Luis (Batangas),None
136,25 Apr 2017 - 12:34 AM,6.22,123.65,19,3.6,060 km S 49° W Kalamansig (Sultan Kudarat),None
137,24 Apr 2017 - 11:40 PM,14.58,120.05,45,2.0,027 km S 63° W Morong (Bataan),None
138,24 Apr 2017 - 11:18 PM,8.41,126.68,33,2.7,038 km N 84° E Hinatuan (Surigao Del Sur),None
176,22 Apr 2017 - 08:44 PM,9.88,125.50,25,2.8,010 km N 05° E Surigao City (Surigao Del Norte),None


In [64]:
df.at[133, "Coordinates"] = "014 km S 25° E"
df.at[133, "Location"] = "Cataingan (Masbate)"

df.at[134, "Coordinates"] = "007 km S 68° E"
df.at[134, "Location"] = "Kitcharao (Agusan Del Norte)"

df.at[135, "Coordinates"] = "008 km N 52° W"
df.at[135, "Location"] = "San Luis (Batangas)"

df.at[136, "Coordinates"] = "060 km S 49° W"
df.at[136, "Location"] = "Kalamansig (Sultan Kudarat)"

df.at[137, "Coordinates"] = "027 km S 63° W"
df.at[137, "Location"] = "Morong (Bataan)"

df.at[138, "Coordinates"] = "038 km N 84° E"
df.at[138, "Location"] = "Hinatuan (Surigao Del Sur)"


df.at[176, "Coordinates"] = "010 km N 05° E"
df.at[176, "Location"] = "Surigao City (Surigao Del Norte)"

In [67]:

df.to_csv("phivolcs_earthquakes_2017_04_April.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())

Data Saved: 939 rows


,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Apr 2017 - 11:42 PM,5.66,125.11,3,2.0,024 km S 28° E,Maasim (Sarangani)
1,30 Apr 2017 - 10:51 PM,9.73,122.18,27,3.4,025 km S 83° W,Sipalay (Negros Occidental)
2,30 Apr 2017 - 10:45 PM,7.45,124.50,71,2.0,023 km S 80° W,Banisilan (North Cotabato)
3,30 Apr 2017 - 10:33 PM,7.58,124.66,4,2.3,013 km S 57° W,Wao (Lanao Del Sur)
4,30 Apr 2017 - 10:01 PM,5.80,125.07,6,2.9,010 km S 44° E,Maasim (Sarangani)


#May 2017

In [68]:
import re
import pandas as pd

x  = "/content/2017may.txt"
y = "phivolcs_earthquakes_2017_05_May.csv"

#patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

records = []
state = 0#for header
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        if state == 0:
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue
        if state == 1:# latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue
        if state == 2:# longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0#reset here after format break
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:#magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            if line:
              cur["Location"] = line
              records.append(cur)
              state = 0
              cur = {}


df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])


for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
#--separation
df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]
#---

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())

#----save---#
df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {OUTPUT_FILE}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         0
dtype: int64
Data Saved: 583 rows → phivolcs_earthquakes(2017_January).csv


In [69]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [70]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,31 May 2017 - 11:19 PM,7.69,124.65,3,3.3,013 km N 65° W,Wao (Lanao Del Sur)
1,31 May 2017 - 10:27 PM,13.57,121.65,27,2.0,024 km N 57° W,Boac (Marinduque)
2,31 May 2017 - 09:40 PM,13.01,124.52,35,2.0,036 km S 83° E,Prieto Diaz (Sorsogon)
3,31 May 2017 - 09:15 PM,11.17,121.60,20,2.5,048 km N 87° W,Laua-an (Antique)
4,31 May 2017 - 07:22 PM,11.01,125.59,25,3.0,015 km S 81° E,Guiuan (Eastern Samar)
5,31 May 2017 - 05:55 PM,4.69,126.14,123,3.7,110 km S 42° E,Sarangani (Davao Occidental)
6,31 May 2017 - 10:26 AM,5.89,123.95,568,3.7,045 km S 37° W,Palimbag (Sultan Kudarat)
7,31 May 2017 - 09:33 AM,5.72,125.05,22,2.5,016 km S 15° E,Maasim (Sarangani)
8,31 May 2017 - 08:37 AM,5.58,124.97,17,2.1,031 km S 07° W,Maasim (Sarangani)
9,31 May 2017 - 08:05 AM,9.73,124.54,24,2.2,004 km S 63° W,Anda (Bohol)


In [ ]:
'''
df.at[133, "Coordinates"] = "014 km S 25° E"
df.at[133, "Location"] = "Cataingan (Masbate)"
'''

In [ ]:
'''
df.to_csv("phivolcs_earthquakes_2017_04_April.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())
'''

Data Saved: 939 rows


,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Apr 2017 - 11:42 PM,5.66,125.11,3,2.0,024 km S 28° E,Maasim (Sarangani)
1,30 Apr 2017 - 10:51 PM,9.73,122.18,27,3.4,025 km S 83° W,Sipalay (Negros Occidental)
2,30 Apr 2017 - 10:45 PM,7.45,124.50,71,2.0,023 km S 80° W,Banisilan (North Cotabato)
3,30 Apr 2017 - 10:33 PM,7.58,124.66,4,2.3,013 km S 57° W,Wao (Lanao Del Sur)
4,30 Apr 2017 - 10:01 PM,5.80,125.07,6,2.9,010 km S 44° E,Maasim (Sarangani)


#June 2017

In [71]:
import re
import pandas as pd

x  = "/content/2017jun.txt"
y = "phivolcs_earthquakes_2017_06_June.csv"

#patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

records = []
state = 0#for header
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        if state == 0:
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue
        if state == 1:# latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue
        if state == 2:# longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0#reset here after format break
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:#magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            if line:
              cur["Location"] = line
              records.append(cur)
              state = 0
              cur = {}


df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])


for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
#--separation
df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]
#---

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())

#----save---#
df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {OUTPUT_FILE}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         2
dtype: int64
Data Saved: 408 rows → phivolcs_earthquakes(2017_January).csv


In [72]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
231,14 Jun 2017 - 11:24 PM,10.69,125.00,1,2.9,006 km S 17° W Abuyog (Leyte),None
338,07 Jun 2017 - 07:21 PM,5.09,127.62,85,3.1,239 km S 68° E oF Jose Abad Santos (Davao Occi...,None


In [76]:

df.at[231	, "Coordinates"] = "006 km S 17° W"
df.at[231	, "Location"] = "Abuyog (Leyte)"

df.at[338	, "Coordinates"] = "239 km S 68° E"
df.at[338	, "Location"] = "Jose Abad Santos (Davao Occidental)"


In [77]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [78]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Jun 2017 - 11:34 PM,11.06,121.84,22,2.0,020 km N 51° W,Patnongon (Antique)
1,30 Jun 2017 - 06:57 PM,17.87,120.40,21,2.4,006 km S 88° W,Sinait (Ilocos Norte)
2,30 Jun 2017 - 12:17 PM,13.59,119.98,32,2.4,032 km S 64° W,Looc (Occidental Mindoro)
3,30 Jun 2017 - 11:39 AM,10.02,124.15,2,2.9,010 km N 44° W,Danao (Bohol)
4,30 Jun 2017 - 10:39 AM,19.93,121.74,24,2.5,047 km S 17° W,Sabtang (Batanes)
5,30 Jun 2017 - 08:44 AM,6.45,123.72,1,3.3,040 km S 69° W,Kalamansig (Sultan Kudarat)
6,30 Jun 2017 - 07:32 AM,16.56,120.46,17,2.4,005 km N 24° E,Burgos (La Union)
7,30 Jun 2017 - 02:48 AM,8.11,126.00,20,2.5,007 km S 20° E,Bunawan (Agusan Del Sur)
8,30 Jun 2017 - 02:06 AM,18.01,120.05,28,2.7,046 km N 79° W,Badoc (Ilocos Norte)
9,30 Jun 2017 - 12:34 AM,15.05,120.16,69,2.2,009 km N 03° E,San Marcelino (Zambales)


In [79]:

df.to_csv("phivolcs_earthquakes_2017_06_June.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())


Data Saved: 408 rows


,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Jun 2017 - 11:34 PM,11.06,121.84,22,2.0,020 km N 51° W,Patnongon (Antique)
1,30 Jun 2017 - 06:57 PM,17.87,120.40,21,2.4,006 km S 88° W,Sinait (Ilocos Norte)
2,30 Jun 2017 - 12:17 PM,13.59,119.98,32,2.4,032 km S 64° W,Looc (Occidental Mindoro)
3,30 Jun 2017 - 11:39 AM,10.02,124.15,2,2.9,010 km N 44° W,Danao (Bohol)
4,30 Jun 2017 - 10:39 AM,19.93,121.74,24,2.5,047 km S 17° W,Sabtang (Batanes)


#July 2017 (added condition)

In [93]:
import re
import pandas as pd

x  = "/content/2017jul.txt"
y = "phivolcs_earthquakes_2017_07_July.csv"

#added patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

#added
row_rx = re.compile(
    r"^(?P<datetime>\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM))\s+"
    r"(?P<lat>\d{1,2}\.\d{2})\s+"
    r"(?P<long>\d{1,3}\.\d{2})\s+"
    r"(?P<depth>\d+)\s+"
    r"(?P<mag>\d+\.\d)$"
)

records = []
state = 0
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        m = row_rx.match(line)
        if m:
            cur = {
                "Date - Time (Philippine Time)": m.group("datetime"),
                "Latitude (ºN)": m.group("lat"),
                "Longitude (ºE)": m.group("long"),
                "Depth (km)": m.group("depth"),
                "Mag": m.group("mag"),
            }
            state = 5
            continue


        if state == 0:  # datetime
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue

        if state == 1:  # latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue

        if state == 2:  # longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:  # magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            cur["Location"] = line if line else None
            records.append(cur)
            state = 0
            cur = {}

df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])

for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())


df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {y}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         0
dtype: int64
Data Saved: 521 rows → phivolcs_earthquakes_2017_07_July.csv


In [94]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [95]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,31 Jul 2017 - 11:52 PM,8.05,126.29,95,3.5,013 km N 85° W,Lingig (Surigao Del Sur)
1,31 Jul 2017 - 11:42 PM,13.20,124.08,28,1.8,005 km N 72° W,Rapu-rapu (Albay)
2,31 Jul 2017 - 11:27 PM,8.05,127.06,12,3.4,071 km N 89° E,Lingig (Surigao Del Sur)
3,31 Jul 2017 - 06:44 PM,8.73,126.60,33,2.7,035 km S 76° E,Marihatag (Surigao Del Sur)
4,31 Jul 2017 - 11:14 AM,9.87,125.44,27,2.3,011 km N 35° W,Surigao City (Surigao Del Norte)
5,31 Jul 2017 - 06:57 AM,14.04,123.89,28,2.0,026 km N 51° E,Garchitorena (Camarines Sur)
6,31 Jul 2017 - 01:13 AM,17.36,120.35,12,2.6,010 km N 71° W,San Esteban (Ilocos Sur)
7,30 Jul 2017 - 11:30 PM,13.06,123.98,7,3.1,010 km N 13° W,Sorsogon (Sorsogon)
8,30 Jul 2017 - 10:56 PM,9.90,126.15,14,2.9,008 km S 60° E,San Isidro (Surigao Del Norte)
9,30 Jul 2017 - 08:28 PM,5.50,126.39,196,2.7,095 km S 64° E,Jose Abad Santos (Davao Occidental)


In [96]:
'''
df.at[133, "Coordinates"] = "014 km S 25° E"
df.at[133, "Location"] = "Cataingan (Masbate)"
'''

'\ndf.at[133, "Coordinates"] = "014 km S 25° E"\ndf.at[133, "Location"] = "Cataingan (Masbate)"\n'

In [97]:
'''
df.to_csv("phivolcs_earthquakes_2017_04_April.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())
'''

'\ndf.to_csv("phivolcs_earthquakes_2017_04_April.csv", index=False, encoding="utf-8")\nprint(f"Data Saved: {len(df)} rows")\ndisplay(df.head())\n'

#August 2017 (added condition)

In [98]:
import re
import pandas as pd

x  = "/content/2017aug.txt"
y = "phivolcs_earthquakes_2017_08_August.csv"

#added patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

#added
row_rx = re.compile(
    r"^(?P<datetime>\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM))\s+"
    r"(?P<lat>\d{1,2}\.\d{2})\s+"
    r"(?P<long>\d{1,3}\.\d{2})\s+"
    r"(?P<depth>\d+)\s+"
    r"(?P<mag>\d+\.\d)$"
)

records = []
state = 0
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        m = row_rx.match(line)
        if m:
            cur = {
                "Date - Time (Philippine Time)": m.group("datetime"),
                "Latitude (ºN)": m.group("lat"),
                "Longitude (ºE)": m.group("long"),
                "Depth (km)": m.group("depth"),
                "Mag": m.group("mag"),
            }
            state = 5
            continue


        if state == 0:  # datetime
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue

        if state == 1:  # latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue

        if state == 2:  # longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:  # magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            cur["Location"] = line if line else None
            records.append(cur)
            state = 0
            cur = {}

df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])

for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()
cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())


df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {y}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         6
dtype: int64
Data Saved: 421 rows → phivolcs_earthquakes_2017_08_August.csv


In [99]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
46,29 Aug 2017 - 05:56 PM,6.63,125.82,74,2.7,028 km S 85° W Governor Generoso (Davao Oriental),None
47,29 Aug 2017 - 04:08 PM,13.66,124.86,12,3.0,040 km S 49° E Baras (Catanduanes),None
48,29 Aug 2017 - 04:03 PM,5.29,126.94,14,2.9,161 km S 66° E Jose Abad Santos (Davao Occiden...,None
49,29 Aug 2017 - 02:11 PM,10.08,124.99,17,4.0,016 km S 56° W Liloan (Southern Leyte),None
50,29 Aug 2017 - 06:49 AM,14.89,121.96,14,2.0,005 km N 08° W Burdeos (Quezon),None
114,24 Aug 2017 - 11:10 AM,13.83,119.95,10,2.5,035 km N 71° W Looc (Occidental Mindoro),None


In [104]:
fix = {
    46: {"Coordinates": "028 km S 85° W", "Location": "Governor Generoso (Davao Oriental)"},
    47: {"Coordinates": "040 km S 49° E", "Location": "Baras (Catanduanes)"},
    48: {"Coordinates": "161 km S 66° E", "Location": "Jose Abad Santos (Davao Occidental)"},
    49: {"Coordinates": "016 km S 56° W", "Location": "Liloan (Southern Leyte)"},
    50: {"Coordinates": "005 km N 08° W", "Location": "Burdeos (Quezon)"},
    114: {"Coordinates": "035 km N 71° W", "Location": "Looc (Occidental Mindoro)"}
}

for idx, changes in fix.items():
    for col, val in changes.items():
        df.at[idx, col] = val



In [105]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [106]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,31 Aug 2017 - 09:55 PM,11.27,124.62,7,3.2,004 km S 46° W,Capoocan (Leyte)
1,31 Aug 2017 - 08:08 PM,13.21,122.43,7,2.7,018 km S 33° W,San Francisco (Aurora) (Quezon)
2,31 Aug 2017 - 07:16 PM,9.76,124.74,1,3.0,018 km N 85° E,Anda (Bohol)
3,31 Aug 2017 - 03:06 PM,15.04,119.88,28,2.0,020 km S 85° W,San Felipe (Zambales)
4,31 Aug 2017 - 08:15 AM,9.86,124.04,5,3.3,004 km N 85° E,Catigbian (Bohol)
5,31 Aug 2017 - 07:44 AM,6.77,123.89,32,1.9,023 km N 55° W,Lebak (Sultan Kudarat)
6,31 Aug 2017 - 06:57 AM,14.34,120.27,56,3.1,026 km S 65° W,Mariveles (Bataan)
7,31 Aug 2017 - 05:47 AM,18.28,122.82,5,2.7,075 km S 76° E,Santa Ana (Cagayan)
8,31 Aug 2017 - 05:24 AM,18.11,120.44,29,2.2,015 km N 65° W,Batac (Ilocos Norte)
9,31 Aug 2017 - 05:06 AM,12.70,123.41,3,2.2,031 km S 42° W,Donsol (Sorsogon)


In [107]:

df.to_csv("phivolcs_earthquakes_2017_08_August.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())


Data Saved: 421 rows


,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,31 Aug 2017 - 09:55 PM,11.27,124.62,7,3.2,004 km S 46° W,Capoocan (Leyte)
1,31 Aug 2017 - 08:08 PM,13.21,122.43,7,2.7,018 km S 33° W,San Francisco (Aurora) (Quezon)
2,31 Aug 2017 - 07:16 PM,9.76,124.74,1,3.0,018 km N 85° E,Anda (Bohol)
3,31 Aug 2017 - 03:06 PM,15.04,119.88,28,2.0,020 km S 85° W,San Felipe (Zambales)
4,31 Aug 2017 - 08:15 AM,9.86,124.04,5,3.3,004 km N 85° E,Catigbian (Bohol)


#September 2017 (added condition)

In [157]:
import re
import pandas as pd

x  = "/content/2017sep.txt"
y = "phivolcs_earthquakes_2017_09_September.csv"

#added patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

#added
row_rx = re.compile(
    r"^(?P<datetime>\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM))\s+"
    r"(?P<lat>\d{1,2}\.\d{2})\s+"
    r"(?P<long>\d{1,3}\.\d{2})\s+"
    r"(?P<depth>\d+)\s+"
    r"(?P<mag>\d+\.\d)$"
)

records = []
state = 0
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        m = row_rx.match(line)
        if m:
            cur = {
                "Date - Time (Philippine Time)": m.group("datetime"),
                "Latitude (ºN)": m.group("lat"),
                "Longitude (ºE)": m.group("long"),
                "Depth (km)": m.group("depth"),
                "Mag": m.group("mag"),
            }
            state = 5
            continue


        if state == 0:  # datetime
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue

        if state == 1:  # latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue

        if state == 2:  # longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:  # magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            cur["Location"] = line if line else None
            records.append(cur)
            state = 0
            cur = {}

df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])

for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()

cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())


df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {y}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)     0
Latitude (ºN)                     0
Longitude (ºE)                    0
Depth (km)                        0
Mag                               0
Coordinates                       0
Location                         26
dtype: int64
Data Saved: 361 rows → phivolcs_earthquakes_2017_09_September.csv


In [158]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
193,15 Sep 2017 - 10:20 PM,7.73,126.94,16,3.0,046 km N 68° E Baganga (Davao Oriental),None
194,15 Sep 2017 - 05:38 PM,18.40,120.29,26,2.7,039 km N 54° W Laoag City (Ilocos Norte),None
195,15 Sep 2017 - 04:29 PM,14.04,120.35,17,2.1,030 km S 82° W Nasugbu (Batangas),None
196,15 Sep 2017 - 03:44 PM,6.31,126.24,15,3.0,042 km S 26° E Governor Generoso (Davao Oriental),None
197,15 Sep 2017 - 02:51 PM,11.16,124.70,10,3.0,015 km S 80° E Kananga (Leyte),None
198,15 Sep 2017 - 11:11 AM,10.35,125.33,1,2.8,015 km S 71° E Hinunangan (Southern Leyte),None
199,15 Sep 2017 - 07:14 AM,9.38,126.26,49,3.1,014 km N 33° E Cortes (Surigao Del Sur),None
200,15 Sep 2017 - 06:43 AM,16.13,120.16,6,2.9,010 km N 48° E Sual (Pangasinan),None
201,15 Sep 2017 - 04:26 AM,8.90,126.28,35,2.8,003 km S 54° W Cagwait (Surigao Del Sur),None
202,15 Sep 2017 - 03:27 AM,10.97,124.73,1,2.6,007 km N 34° E Albuera (Leyte),None


In [159]:

fix = {
    193: {"Coordinates": "046 km N 68° E", "Location": "Baganga (Davao Oriental)"},
    194: {"Coordinates": "039 km N 54° W", "Location": "Laoag City (Ilocos Norte)"},
    195: {"Coordinates": "030 km S 82° W", "Location": "Nasugbu (Batangas)"},
    196: {"Coordinates": "042 km S 26° E", "Location": "Governor Generoso (Davao Oriental)"},
    197: {"Coordinates": "015 km S 80° E", "Location": "Kananga (Leyte)"},
    198: {"Coordinates": "015 km S 71° E", "Location": "Hinunangan (Southern Leyte)"},
    199: {"Coordinates": "014 km N 33° E", "Location": "Cortes (Surigao Del Sur)"},
    200: {"Coordinates": "010 km N 48° E", "Location": "Sual (Pangasinan)"},
    201: {"Coordinates": "003 km S 54° W", "Location": "Cagwait (Surigao Del Sur)"},
    202: {"Coordinates": "007 km N 34° E", "Location": "Albuera (Leyte)"},
    203: {"Coordinates": "004 km S 48° E", "Location": "Hinundayan (Southern Leyte)"},
    204: {"Coordinates": "142 km S 50° E", "Location": "Governor Generoso (Davao Oriental)"},
    205: {"Coordinates": "048 km N 67° W", "Location": "Lebak (Sultan Kudarat)"},
    206: {"Coordinates": "015 km N 79° W", "Location": "Siocon (Zamboanga Del Norte)"},
    207: {"Coordinates": "032 km N 78° E", "Location": "Looc (Occidental Mindoro)"},
    208: {"Coordinates": "026 km N 14° E", "Location": "Bobon (Northern Samar)"},
    209: {"Coordinates": "036 km N 23° E", "Location": "Claveria (Cagayan)"},
    210: {"Coordinates": "004 km N 81° W", "Location": "Danao (Bohol)"},
    211: {"Coordinates": "015 km S 29° W", "Location": "San Andres (Quezon)"},
    212: {"Coordinates": "006 km S 62° W", "Location": "Alegria (Surigao Del Norte)"},
    213: {"Coordinates": "064 km S 72° W", "Location": "Upi (Maguindanao)"},
    214: {"Coordinates": "109 km S 84° W", "Location": "Palauig (Zambales)"},
    215: {"Coordinates": "016 km S 14° W", "Location": "Donsol (Sorsogon)"},
    216: {"Coordinates": "004 km S 86° E", "Location": "Samal (Island Garden City Of Samal)"},
    217: {"Coordinates": "016 km S 30° W", "Location": "Pilar (Sorsogon)"},
    218: {"Coordinates": "016 km S 82° W", "Location": "Dinagat (Dinagat)"}
}


for idx, changes in fix.items():
    for col, val in changes.items():
        df.at[idx, col] = val



In [160]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [161]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Sep 2017 - 09:42 PM,9.60,123.23,18,2.2,009 km N 70° W,Ginatilan (Cebu)
1,30 Sep 2017 - 10:45 AM,7.19,126.63,28,2.8,011 km S 75° E,Manay (Davao Oriental)
2,30 Sep 2017 - 07:18 AM,13.63,120.52,78,3.2,025 km S 30° W,Calatagan (Batangas)
3,30 Sep 2017 - 02:00 AM,8.46,124.83,2,3.2,010 km S 32° W,Malitbog (Bukidnon)
4,30 Sep 2017 - 01:12 AM,7.53,124.74,1,2.5,013 km S 07° W,Wao (Lanao del Sur)
5,30 Sep 2017 - 12:51 AM,9.41,126.14,18,2.8,016 km N 23° W,Cortes (Surigao del Sur)
6,30 Sep 2017 - 12:46 AM,7.66,124.74,1,3.4,002 km N 34° W,Wao (Lanao Del Sur)
7,29 Sep 2017 - 06:16 PM,6.10,124.95,16,2.3,018 km N 39° W,Polomok (South Cotabato)
8,29 Sep 2017 - 04:03 PM,9.62,125.42,1,2.6,002 km N82° W,Malimono (Surigao Del Norte)
9,29 Sep 2017 - 03:40 PM,9.98,122.88,32,2.1,007 km S 85° E,Kabankalan City (Negros Occidental)


In [162]:

df.to_csv("phivolcs_earthquakes_2017_09_September.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())


Data Saved: 361 rows


,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Sep 2017 - 09:42 PM,9.60,123.23,18,2.2,009 km N 70° W,Ginatilan (Cebu)
1,30 Sep 2017 - 10:45 AM,7.19,126.63,28,2.8,011 km S 75° E,Manay (Davao Oriental)
2,30 Sep 2017 - 07:18 AM,13.63,120.52,78,3.2,025 km S 30° W,Calatagan (Batangas)
3,30 Sep 2017 - 02:00 AM,8.46,124.83,2,3.2,010 km S 32° W,Malitbog (Bukidnon)
4,30 Sep 2017 - 01:12 AM,7.53,124.74,1,2.5,013 km S 07° W,Wao (Lanao del Sur)


#October 2017 (added condition)

In [141]:
import re
import pandas as pd

x  = "/content/2017oct.txt"
y = "phivolcs_earthquakes_2017_10_October.csv"

#added patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

#added
row_rx = re.compile(
    r"^(?P<datetime>\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM))\s+"
    r"(?P<lat>\d{1,2}\.\d{2})\s+"
    r"(?P<long>\d{1,3}\.\d{2})\s+"
    r"(?P<depth>\d+)\s+"
    r"(?P<mag>\d+\.\d)$"
)

records = []
state = 0
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        m = row_rx.match(line)
        if m:
            cur = {
                "Date - Time (Philippine Time)": m.group("datetime"),
                "Latitude (ºN)": m.group("lat"),
                "Longitude (ºE)": m.group("long"),
                "Depth (km)": m.group("depth"),
                "Mag": m.group("mag"),
            }
            state = 5
            continue


        if state == 0:  # datetime
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue

        if state == 1:  # latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue

        if state == 2:  # longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:  # magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            cur["Location"] = line if line else None
            records.append(cur)
            state = 0
            cur = {}

df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])

for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()

cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())


df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {y}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         0
dtype: int64
Data Saved: 330 rows → phivolcs_earthquakes_2017_10_October.csv


In [142]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [143]:
'''
fix = {
    193: {"Coordinates": "046 km N 68° E", "Location": "Baganga (Davao Oriental)"},
    194: {"Coordinates": "039 km N 54° W", "Location": "Laoag City (Ilocos Norte)"},
    195: {"Coordinates": "030 km S 82° W", "Location": "Nasugbu (Batangas)"},
    196: {"Coordinates": "042 km S 26° E", "Location": "Governor Generoso (Davao Oriental)"},
    197: {"Coordinates": "015 km S 80° E", "Location": "Kananga (Leyte)"},
    198: {"Coordinates": "015 km S 71° E", "Location": "Hinunangan (Southern Leyte)"},
    199: {"Coordinates": "014 km N 33° E", "Location": "Cortes (Surigao Del Sur)"},
    200: {"Coordinates": "010 km N 48° E", "Location": "Sual (Pangasinan)"},
    201: {"Coordinates": "003 km S 54° W", "Location": "Cagwait (Surigao Del Sur)"},
    202: {"Coordinates": "007 km N 34° E", "Location": "Albuera (Leyte)"},
    203: {"Coordinates": "004 km S 48° E", "Location": "Hinundayan (Southern Leyte)"},
    204: {"Coordinates": "142 km S 50° E", "Location": "Governor Generoso (Davao Oriental)"},
    205: {"Coordinates": "048 km N 67° W", "Location": "Lebak (Sultan Kudarat)"},
    206: {"Coordinates": "015 km N 79° W", "Location": "Siocon (Zamboanga Del Norte)"},
    207: {"Coordinates": "032 km N 78° E", "Location": "Looc (Occidental Mindoro)"},
    208: {"Coordinates": "026 km N 14° E", "Location": "Bobon (Northern Samar)"},
    209: {"Coordinates": "036 km N 23° E", "Location": "Claveria (Cagayan)"},
    210: {"Coordinates": "004 km N 81° W", "Location": "Danao (Bohol)"},
    211: {"Coordinates": "015 km S 29° W", "Location": "San Andres (Quezon)"},
    212: {"Coordinates": "006 km S 62° W", "Location": "Alegria (Surigao Del Norte)"},
    213: {"Coordinates": "064 km S 72° W", "Location": "Upi (Maguindanao)"},
    214: {"Coordinates": "109 km S 84° W", "Location": "Palauig (Zambales)"},
    215: {"Coordinates": "016 km S 14° W", "Location": "Donsol (Sorsogon)"},
    216: {"Coordinates": "004 km S 86° E", "Location": "Samal (Island Garden City Of Samal)"},
    217: {"Coordinates": "016 km S 30° W", "Location": "Pilar (Sorsogon)"},
    218: {"Coordinates": "016 km S 82° W", "Location": "Dinagat (Dinagat)"}
}


for idx, changes in fix.items():
    for col, val in changes.items():
        df.at[idx, col] = val

'''

'\nfix = {\n    193: {"Coordinates": "046 km N 68° E", "Location": "Baganga (Davao Oriental)"},\n    194: {"Coordinates": "039 km N 54° W", "Location": "Laoag City (Ilocos Norte)"},\n    195: {"Coordinates": "030 km S 82° W", "Location": "Nasugbu (Batangas)"},\n    196: {"Coordinates": "042 km S 26° E", "Location": "Governor Generoso (Davao Oriental)"},\n    197: {"Coordinates": "015 km S 80° E", "Location": "Kananga (Leyte)"},\n    198: {"Coordinates": "015 km S 71° E", "Location": "Hinunangan (Southern Leyte)"},\n    199: {"Coordinates": "014 km N 33° E", "Location": "Cortes (Surigao Del Sur)"},\n    200: {"Coordinates": "010 km N 48° E", "Location": "Sual (Pangasinan)"},\n    201: {"Coordinates": "003 km S 54° W", "Location": "Cagwait (Surigao Del Sur)"},\n    202: {"Coordinates": "007 km N 34° E", "Location": "Albuera (Leyte)"},\n    203: {"Coordinates": "004 km S 48° E", "Location": "Hinundayan (Southern Leyte)"},\n    204: {"Coordinates": "142 km S 50° E", "Location": "Governor G

In [144]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [145]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,31 Oct 2017 - 11:35 PM,5.54,126.54,107,3.2,110 km S 69° E,Jose Abad Santos (Davao Occidental)
1,31 Oct 2017 - 10:25 PM,12.94,124.60,23,2.2,038 km N 42° E,Biri (Northern Samar)
2,31 Oct 2017 - 09:12 PM,13.09,120.87,1,2.3,016 km N 83° E,Santa Cruz (Occidental Mindoro)
3,31 Oct 2017 - 06:14 PM,8.54,123.87,8,2.6,026 km S 78° E,Dipolog City
4,31 Oct 2017 - 03:45 PM,16.90,120.24,25,2.6,015 km N 72° W,Luna (La Union)
5,31 Oct 2017 - 11:14 AM,14.59,124.00,19,4.5,062 km N 17° W,Pandan (Cataduanes)
6,31 Oct 2017 - 11:11 AM,3.79,126.39,113,4.3,209 km S 29° E,Sarangani (Davao Occidental)
7,31 Oct 2017 - 04:51 AM,4.88,124.37,105,3.2,126 km S 13° W,Kiamba (Sarangani)
8,31 Oct 2017 - 01:49 AM,13.80,120.55,122,3.0,010 km S 70° W,Calatagan (Batangas)
9,31 Oct 2017 - 01:25 AM,13.72,120.52,93,3.1,017 km S 45° W,Calatagan (Batangas)


In [146]:
'''
df.to_csv("phivolcs_earthquakes_2017_09_September.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())
'''

'\ndf.to_csv("phivolcs_earthquakes_2017_09_September.csv", index=False, encoding="utf-8")\nprint(f"Data Saved: {len(df)} rows")\ndisplay(df.head())\n'

#November2017 (added condition)

In [163]:
import re
import pandas as pd

x  = "/content/2017nov.txt"
y = "phivolcs_earthquakes_2017_11_November.csv"

#added patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

#added
row_rx = re.compile(
    r"^(?P<datetime>\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM))\s+"
    r"(?P<lat>\d{1,2}\.\d{2})\s+"
    r"(?P<long>\d{1,3}\.\d{2})\s+"
    r"(?P<depth>\d+)\s+"
    r"(?P<mag>\d+\.\d)$"
)

records = []
state = 0
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        m = row_rx.match(line)
        if m:
            cur = {
                "Date - Time (Philippine Time)": m.group("datetime"),
                "Latitude (ºN)": m.group("lat"),
                "Longitude (ºE)": m.group("long"),
                "Depth (km)": m.group("depth"),
                "Mag": m.group("mag"),
            }
            state = 5
            continue


        if state == 0:  # datetime
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue

        if state == 1:  # latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue

        if state == 2:  # longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:  # magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            cur["Location"] = line if line else None
            records.append(cur)
            state = 0
            cur = {}

df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])

for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()

cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())


df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {y}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         4
dtype: int64
Data Saved: 382 rows → phivolcs_earthquakes_2017_11_November.csv


In [164]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
216,12 Nov 2017 - 01:08 PM,13.85,120.01,33,2.0,004 km S 71° E Santa Teresita (Batangas),None
217,12 Nov 2017 - 12:39 PM,19.19,121.13,28,3.1,036 km S 74° W Calayan (Cagayan),None
218,12 Nov 2017 - 12:29 PM,13.53,123.66,13,2.7,021 km N 55° E Buhi (Camarines Sur),None
219,12 Nov 2017 - 09:52 AM,16.81,122.61,13,2.5,033 km S 34° E Palanan (Isabela),None


In [165]:

fix = {
    216: {"Coordinates": "004 km S 71° E", "Location": "Santa Teresita (Batangas)"},
    217: {"Coordinates": "036 km S 74° W", "Location": "Calayan (Cagayan)"},
    218: {"Coordinates": "021 km N 55° E", "Location": "Buhi (Camarines Sur)"},
    219: {"Coordinates": "033 km S 34° E", "Location": "Palanan (Isabela)"},

}


for idx, changes in fix.items():
    for col, val in changes.items():
        df.at[idx, col] = val



In [166]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [167]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Nov 2017 - 11:15 PM,10.10,123.38,4,2.3,002 km S 16° E,Guihulngan (Negros Oriental)
1,30 Nov 2017 - 09:50 PM,9.40,122.54,20,2.4,011 km S 85° W,Basay (Negros Oriental)
2,30 Nov 2017 - 09:38 PM,14.95,123.97,21,2.3,102 km N 12° W,Pandan (Catanduanes)
3,30 Nov 2017 - 09:15 PM,3.61,125.67,185,3.3,203 km S 06° E,Sarangani (Davao Occidental)
4,30 Nov 2017 - 08:54 PM,18.31,120.46,30,2.3,017 km S 81° W,Pasuquin (Ilocos Norte)
5,30 Nov 2017 - 08:52 PM,8.05,126.48,11,2.4,007 km N 81° E,Lingig (Surigao Del Sur)
6,30 Nov 2017 - 07:46 PM,16.66,120.61,1,2.0,011 km N 36° W,Atok (Benguet)
7,30 Nov 2017 - 06:54 PM,18.09,120.70,36,1.9,002 km S 04° E,Dingras (Ilocos Norte)
8,30 Nov 2017 - 05:58 PM,10.09,123.27,8,2.8,004 km S 08° W,Guihulngan (Negros Oriental)
9,30 Nov 2017 - 01:59 PM,18.56,120.96,12,4.3,018 km S 85° E,Pagudpud (Ilocos Norte)


In [168]:

df.to_csv("phivolcs_earthquakes_2017_11_November.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())


Data Saved: 382 rows


,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,30 Nov 2017 - 11:15 PM,10.10,123.38,4,2.3,002 km S 16° E,Guihulngan (Negros Oriental)
1,30 Nov 2017 - 09:50 PM,9.40,122.54,20,2.4,011 km S 85° W,Basay (Negros Oriental)
2,30 Nov 2017 - 09:38 PM,14.95,123.97,21,2.3,102 km N 12° W,Pandan (Catanduanes)
3,30 Nov 2017 - 09:15 PM,3.61,125.67,185,3.3,203 km S 06° E,Sarangani (Davao Occidental)
4,30 Nov 2017 - 08:54 PM,18.31,120.46,30,2.3,017 km S 81° W,Pasuquin (Ilocos Norte)


#December2017 (added condition)

In [169]:
import re
import pandas as pd

x  = "/content/2017dec.txt"
y = "phivolcs_earthquakes_2017_12_December.csv"

#added patterns
dt_rx   = re.compile(r"^\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM)$", re.I)
lat_rx  = re.compile(r"^\d{1,2}\.\d{2}$")
lon_rx  = re.compile(r"^\d{1,3}\.\d{2}$")
dep_rx  = re.compile(r"^\d+$")
mag_rx  = re.compile(r"^\d+(?:\.\d)?$")
month_rx = re.compile(r"^(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)$", re.I)

#added
row_rx = re.compile(
    r"^(?P<datetime>\d{1,2}\s+\w+\s+\d{4}\s*-\s*\d{1,2}:\d{2}\s*(?:AM|PM))\s+"
    r"(?P<lat>\d{1,2}\.\d{2})\s+"
    r"(?P<long>\d{1,3}\.\d{2})\s+"
    r"(?P<depth>\d+)\s+"
    r"(?P<mag>\d+\.\d)$"
)

records = []
state = 0
cur = {}

def header_line(s: str) -> bool:
    hdrs = {
        "date - time (philippine time)",
        "latitude (ºn)", "latitude (°n)", "latitude (n)",
        "longitude (ºe)", "longitude (°e)", "longitude (e)",
        "depth (km)", "mag", "location", "seismicity maps", "description:"
    }
    t = s.strip().lower()
    return (t in hdrs) or bool(month_rx.match(s))

with open(x, encoding="utf-8", errors="ignore") as f:
    for raw in f:
        line = raw.strip()
        if not line or header_line(line):
            continue

        m = row_rx.match(line)
        if m:
            cur = {
                "Date - Time (Philippine Time)": m.group("datetime"),
                "Latitude (ºN)": m.group("lat"),
                "Longitude (ºE)": m.group("long"),
                "Depth (km)": m.group("depth"),
                "Mag": m.group("mag"),
            }
            state = 5
            continue


        if state == 0:  # datetime
            if dt_rx.match(line):
                cur = {"Date - Time (Philippine Time)": line}
                state = 1
            continue

        if state == 1:  # latitude
            if lat_rx.match(line):
                cur["Latitude (ºN)"] = line
                state = 2
            else:
                if dt_rx.match(line):
                    cur = {"Date - Time (Philippine Time)": line}
                    state = 1
            continue

        if state == 2:  # longitude
            if lon_rx.match(line):
                cur["Longitude (ºE)"] = line
                state = 3
            else:
                state = 0
            continue

        if state == 3:  # depth
            if dep_rx.match(line):
                cur["Depth (km)"] = line
                state = 4
            else:
                state = 0
            continue

        if state == 4:  # magnitude
            if mag_rx.match(line):
                cur["Mag"] = line
                state = 5
            else:
                state = 0
            continue

        if state == 5:  # location
            cur["Location"] = line if line else None
            records.append(cur)
            state = 0
            cur = {}

df = pd.DataFrame.from_records(records, columns=[
    "Date - Time (Philippine Time)",
    "Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag", "Location"
])

for col in ["Latitude (ºN)", "Longitude (ºE)", "Depth (km)", "Mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df[["Coordinates", "Location"]] = df["Location"].str.split("of ", n=1, expand=True)
df["Coordinates"] = df["Coordinates"].str.strip()

cols = ["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)","Depth (km)", "Mag", "Coordinates", "Location"]
df = df[cols]

dups_mask = df.duplicated(subset=["Date - Time (Philippine Time)", "Latitude (ºN)", "Longitude (ºE)"], keep=False)
duplicates = df[dups_mask]
print("Duplicates found:\n", duplicates if not duplicates.empty else "None")
print("Dups No:", int(dups_mask.sum()))
print("Nulls per column:\n", df.isnull().sum())


df.to_csv(y, index=False, encoding="utf-8-sig")
print(f"Data Saved: {len(df)} rows → {y}")


Duplicates found:
 None
Dups No: 0
Nulls per column:
 Date - Time (Philippine Time)    0
Latitude (ºN)                    0
Longitude (ºE)                   0
Depth (km)                       0
Mag                              0
Coordinates                      0
Location                         0
dtype: int64
Data Saved: 225 rows → phivolcs_earthquakes_2017_12_December.csv


In [170]:
df[df.isnull().any(axis=1)]

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location


In [171]:
'''
fix = {
    216: {"Coordinates": "004 km S 71° E", "Location": "Santa Teresita (Batangas)"},
    217: {"Coordinates": "036 km S 74° W", "Location": "Calayan (Cagayan)"},
    218: {"Coordinates": "021 km N 55° E", "Location": "Buhi (Camarines Sur)"},
    219: {"Coordinates": "033 km S 34° E", "Location": "Palanan (Isabela)"},

}


for idx, changes in fix.items():
    for col, val in changes.items():
        df.at[idx, col] = val

'''

'\nfix = {\n    216: {"Coordinates": "004 km S 71° E", "Location": "Santa Teresita (Batangas)"},\n    217: {"Coordinates": "036 km S 74° W", "Location": "Calayan (Cagayan)"},\n    218: {"Coordinates": "021 km N 55° E", "Location": "Buhi (Camarines Sur)"},\n    219: {"Coordinates": "033 km S 34° E", "Location": "Palanan (Isabela)"},\n\n}\n\n\nfor idx, changes in fix.items():\n    for col, val in changes.items():\n        df.at[idx, col] = val\n\n'

In [172]:
df.head(20)

,Date - Time (Philippine Time),Latitude (ºN),Longitude (ºE),Depth (km),Mag,Coordinates,Location
0,24 Dec 2017 - 11:10 AM,13.86,120.55,147,2.8,010 km N 71° W,Calatagan (Batangas)
1,24 Dec 2017 - 12:01 AM,12.42,123.74,1,2.1,006 km S 23° W,San Fernando (Masbate)
2,23 Dec 2017 - 11:31 PM,16.01,121.58,33,2.2,023 km S 05° W,Nagtipunan (Quirino)
3,23 Dec 2017 - 07:25 PM,9.33,126.35,14,3.5,018 km N 71° E,Cortes (Surigao Del Sur)
4,23 Dec 2017 - 06:49 PM,8.79,126.33,28,2.5,004 km S 65° E,Marihatag (Surigao Del Sur)
5,23 Dec 2017 - 04:53 PM,13.33,120.38,2,2.2,014 km S 47° W,Paluan (Occidental Mindoro)
6,23 Dec 2017 - 02:31 PM,7.19,127.47,22,3.3,008 km S 72° W,Manay (Davao Oriental)
7,23 Dec 2017 - 01:55 PM,17.33,120.28,26,2.4,017 km N 89° W,San Esteban (Ilocos Sur)
8,23 Dec 2017 - 06:32 AM,18.29,121.02,12,2.0,022 km S 30° E,Adams (Ilocos Norte)
9,23 Dec 2017 - 04:04 AM,7.63,126.66,25,3.0,025 km N 63° E,Baganga (Davao Oriental)


In [173]:
'''
df.to_csv("phivolcs_earthquakes_2017_11_November.csv", index=False, encoding="utf-8")
print(f"Data Saved: {len(df)} rows")
display(df.head())
'''

'\ndf.to_csv("phivolcs_earthquakes_2017_11_November.csv", index=False, encoding="utf-8")\nprint(f"Data Saved: {len(df)} rows")\ndisplay(df.head())\n'